In [1]:
import pandas as pd
import numpy as np

In [2]:
df = pd.read_csv(r"C:\Users\sudhanshu\Downloads\archive\amazon_delivery.csv")

In [3]:
df.shape

(43739, 16)

In [4]:
df.head()

,Order_ID,Agent_Age,Agent_Rating,Store_Latitude,Store_Longitude,Drop_Latitude,Drop_Longitude,Order_Date,Order_Time,Pickup_Time,Weather,Traffic,Vehicle,Area,Delivery_Time,Category
0,ialx566343618,37,4.9,22.745049,75.892471,22.765049,75.912471,2022-03-19,11:30:00,11:45:00,Sunny,High,motorcycle,Urban,120,Clothing
1,akqg208421122,34,4.5,12.913041,77.683237,13.043041,77.813237,2022-03-25,19:45:00,19:50:00,Stormy,Jam,scooter,Metropolitian,165,Electronics
2,njpu434582536,23,4.4,12.914264,77.678400,12.924264,77.688400,2022-03-19,08:30:00,08:45:00,Sandstorms,Low,motorcycle,Urban,130,Sports
3,rjto796129700,38,4.7,11.003669,76.976494,11.053669,77.026494,2022-04-05,18:00:00,18:10:00,Sunny,Medium,motorcycle,Metropolitian,105,Cosmetics
4,zguw716275638,32,4.6,12.972793,80.249982,13.012793,80.289982,2022-03-26,13:30:00,13:45:00,Cloudy,High,scooter,Metropolitian,150,Toys


In [5]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 43739 entries, 0 to 43738
Data columns (total 16 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   Order_ID         43739 non-null  object 
 1   Agent_Age        43739 non-null  int64  
 2   Agent_Rating     43685 non-null  float64
 3   Store_Latitude   43739 non-null  float64
 4   Store_Longitude  43739 non-null  float64
 5   Drop_Latitude    43739 non-null  float64
 6   Drop_Longitude   43739 non-null  float64
 7   Order_Date       43739 non-null  object 
 8   Order_Time       43739 non-null  object 
 9   Pickup_Time      43739 non-null  object 
 10  Weather          43648 non-null  object 
 11  Traffic          43739 non-null  object 
 12  Vehicle          43739 non-null  object 
 13  Area             43739 non-null  object 
 14  Delivery_Time    43739 non-null  int64  
 15  Category         43739 non-null  object 
dtypes: float64(5), int64(2), object(9)
memory usage: 5.3+ MB


## **DATA CLEANING**

In [6]:
# Drop duplicates
df = df.drop_duplicates(subset='Order_ID')

In [7]:
df.isna().sum()

Order_ID            0
Agent_Age           0
Agent_Rating       54
Store_Latitude      0
Store_Longitude     0
Drop_Latitude       0
Drop_Longitude      0
Order_Date          0
Order_Time          0
Pickup_Time         0
Weather            91
Traffic             0
Vehicle             0
Area                0
Delivery_Time       0
Category            0
dtype: int64

In [8]:
df['Agent_Rating'] = df.groupby('Category')['Agent_Rating'].transform(lambda x: x.fillna(x.median()))

In [9]:
df['Weather'] = df['Weather'].fillna("Unknown")

In [10]:
df.isna().sum()

Order_ID           0
Agent_Age          0
Agent_Rating       0
Store_Latitude     0
Store_Longitude    0
Drop_Latitude      0
Drop_Longitude     0
Order_Date         0
Order_Time         0
Pickup_Time        0
Weather            0
Traffic            0
Vehicle            0
Area               0
Delivery_Time      0
Category           0
dtype: int64

## **FEATURE ENGINEERING**

In [11]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 43739 entries, 0 to 43738
Data columns (total 16 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   Order_ID         43739 non-null  object 
 1   Agent_Age        43739 non-null  int64  
 2   Agent_Rating     43739 non-null  float64
 3   Store_Latitude   43739 non-null  float64
 4   Store_Longitude  43739 non-null  float64
 5   Drop_Latitude    43739 non-null  float64
 6   Drop_Longitude   43739 non-null  float64
 7   Order_Date       43739 non-null  object 
 8   Order_Time       43739 non-null  object 
 9   Pickup_Time      43739 non-null  object 
 10  Weather          43739 non-null  object 
 11  Traffic          43739 non-null  object 
 12  Vehicle          43739 non-null  object 
 13  Area             43739 non-null  object 
 14  Delivery_Time    43739 non-null  int64  
 15  Category         43739 non-null  object 
dtypes: float64(5), int64(2), object(9)
memory usage: 5.3+ MB


In [12]:
df['Order_Date'] = pd.to_datetime(df['Order_Date'])

In [13]:
df['Delivery_Time'].describe()

count    43739.000000
mean       124.905645
std         51.915451
min         10.000000
25%         90.000000
50%        125.000000
75%        160.000000
max        270.000000
Name: Delivery_Time, dtype: float64

In [14]:
# Late Delivery Flag
df['is_late'] = df['Delivery_Time'].apply(lambda x: 1 if x > 150 else 0)

In [15]:
# Delivery Hours Column
df['Delivery_hours'] = df['Delivery_Time'] / 60

In [16]:
# Speed Buckets
def speed_buckets(x):
    if x <= 90: 
        return "Fast"
    elif x <= 150:
        return "Standard"
    else:
        return "Slow"

df['delivery_speed'] = df['Delivery_Time'].apply(speed_buckets)

In [17]:
df["is_late"].value_counts(normalize=True) * 100

is_late
0    72.118704
1    27.881296
Name: proportion, dtype: float64

In [18]:
df["delivery_speed"].value_counts()

delivery_speed
Standard    19247
Fast        12297
Slow        12195
Name: count, dtype: int64

In [19]:
# Rating Buckets
df['rating_buckets'] = pd.cut(df['Agent_Rating'], bins=[0,2,3,4,5], labels=["Poor", "Average", "Good", "Excellent"])

In [20]:
df.head()

,Order_ID,Agent_Age,Agent_Rating,Store_Latitude,Store_Longitude,Drop_Latitude,Drop_Longitude,Order_Date,Order_Time,Pickup_Time,Weather,Traffic,Vehicle,Area,Delivery_Time,Category,is_late,Delivery_hours,delivery_speed,rating_buckets
0,ialx566343618,37,4.9,22.745049,75.892471,22.765049,75.912471,2022-03-19,11:30:00,11:45:00,Sunny,High,motorcycle,Urban,120,Clothing,0,2.000000,Standard,Excellent
1,akqg208421122,34,4.5,12.913041,77.683237,13.043041,77.813237,2022-03-25,19:45:00,19:50:00,Stormy,Jam,scooter,Metropolitian,165,Electronics,1,2.750000,Slow,Excellent
2,njpu434582536,23,4.4,12.914264,77.678400,12.924264,77.688400,2022-03-19,08:30:00,08:45:00,Sandstorms,Low,motorcycle,Urban,130,Sports,0,2.166667,Standard,Excellent
3,rjto796129700,38,4.7,11.003669,76.976494,11.053669,77.026494,2022-04-05,18:00:00,18:10:00,Sunny,Medium,motorcycle,Metropolitian,105,Cosmetics,0,1.750000,Standard,Excellent
4,zguw716275638,32,4.6,12.972793,80.249982,13.012793,80.289982,2022-03-26,13:30:00,13:45:00,Cloudy,High,scooter,Metropolitian,150,Toys,0,2.500000,Standard,Excellent


## **EXPLORATORY DATA ANALYSIS**

In [21]:
df.groupby("Traffic")["Delivery_hours"].mean()

Traffic
High       2.157069
Jam        2.462650
Low        1.689243
Medium     2.113994
NaN        2.011172
Name: Delivery_hours, dtype: float64

In [22]:
df['Traffic'].value_counts()

Traffic
Low        14999
Jam        13725
Medium     10628
High        4296
NaN           91
Name: count, dtype: int64

In [23]:
df['Traffic'] = df['Traffic'].fillna("Unknown")

In [24]:
df.isna().sum()

Order_ID            0
Agent_Age           0
Agent_Rating        0
Store_Latitude      0
Store_Longitude     0
Drop_Latitude       0
Drop_Longitude      0
Order_Date          0
Order_Time          0
Pickup_Time         0
Weather             0
Traffic             0
Vehicle             0
Area                0
Delivery_Time       0
Category            0
is_late             0
Delivery_hours      0
delivery_speed      0
rating_buckets     53
dtype: int64

In [25]:
df['Agent_Rating'].min(), df['Agent_Rating'].max()

(np.float64(1.0), np.float64(6.0))

In [26]:
df = df[df['Agent_Rating'].between(1,5)]

In [27]:
# Rating Buckets
df['rating_buckets'] = pd.cut(df['Agent_Rating'], bins=[0,2,3,4,5], labels=["Poor", "Average", "Good", "Excellent"], include_lowest=True)

In [28]:
df['rating_buckets'].isna().sum()

np.int64(0)

In [29]:
df.isna().sum()

Order_ID           0
Agent_Age          0
Agent_Rating       0
Store_Latitude     0
Store_Longitude    0
Drop_Latitude      0
Drop_Longitude     0
Order_Date         0
Order_Time         0
Pickup_Time        0
Weather            0
Traffic            0
Vehicle            0
Area               0
Delivery_Time      0
Category           0
is_late            0
Delivery_hours     0
delivery_speed     0
rating_buckets     0
dtype: int64

In [30]:
df['Traffic'].value_counts()

Traffic
Low        14999
Jam        13725
Medium     10628
High        4296
NaN           38
Name: count, dtype: int64

In [31]:
# Feature Engineering
df['high_traffic_flag'] = df['Traffic'].apply(lambda x: 1 if x in ['High', 'Jam'] else 0)

In [32]:
df['delivery_speed'].value_counts()

delivery_speed
Standard    19222
Fast        12278
Slow        12186
Name: count, dtype: int64

In [33]:
# Feature Engineering
df['slow_delivery_flag'] = df['delivery_speed'].apply(lambda x: 1 if x == "Slow" else 0)

In [34]:
df.head()

,Order_ID,Agent_Age,Agent_Rating,Store_Latitude,Store_Longitude,Drop_Latitude,Drop_Longitude,Order_Date,Order_Time,Pickup_Time,...,Vehicle,Area,Delivery_Time,Category,is_late,Delivery_hours,delivery_speed,rating_buckets,high_traffic_flag,slow_delivery_flag
0,ialx566343618,37,4.9,22.745049,75.892471,22.765049,75.912471,2022-03-19,11:30:00,11:45:00,...,motorcycle,Urban,120,Clothing,0,2.000000,Standard,Excellent,0,0
1,akqg208421122,34,4.5,12.913041,77.683237,13.043041,77.813237,2022-03-25,19:45:00,19:50:00,...,scooter,Metropolitian,165,Electronics,1,2.750000,Slow,Excellent,0,1
2,njpu434582536,23,4.4,12.914264,77.678400,12.924264,77.688400,2022-03-19,08:30:00,08:45:00,...,motorcycle,Urban,130,Sports,0,2.166667,Standard,Excellent,0,0
3,rjto796129700,38,4.7,11.003669,76.976494,11.053669,77.026494,2022-04-05,18:00:00,18:10:00,...,motorcycle,Metropolitian,105,Cosmetics,0,1.750000,Standard,Excellent,0,0
4,zguw716275638,32,4.6,12.972793,80.249982,13.012793,80.289982,2022-03-26,13:30:00,13:45:00,...,scooter,Metropolitian,150,Toys,0,2.500000,Standard,Excellent,0,0


In [36]:
df.columns = df.columns.str.lower()

In [37]:
df.columns

Index(['order_id', 'agent_age', 'agent_rating', 'store_latitude',
       'store_longitude', 'drop_latitude', 'drop_longitude', 'order_date',
       'order_time', 'pickup_time', 'weather', 'traffic', 'vehicle', 'area',
       'delivery_time', 'category', 'is_late', 'delivery_hours',
       'delivery_speed', 'rating_buckets', 'high_traffic_flag',
       'slow_delivery_flag'],
      dtype='object')

In [38]:
from sqlalchemy import create_engine

username = "postgres"
password = "25012004"
host = "localhost"
port = "5433"
database = "Amazon Delivery Analytics"

engine = create_engine(f"postgresql+psycopg2://{username}:{password}@{host}:{port}/{database}")

table_name = "amazon_delivery"
df.to_sql(table_name, engine, if_exists="replace", index=False)

print(f"data succecfully loaded into '{table_name}' in database '{database}'.")

data succecfully loaded into 'amazon_delivery' in database 'Amazon Delivery Analytics'.
